# Condition Number of Sampling Matrices for Integer-Shifted Sinc Functions

This notebook computes the 2-norm condition number $\kappa_2(A^{(d)})$ of both:
1. **Pure time sampling matrix** $A_{\mathrm{time}}^{(d)}$ (all $d$ samples in time domain)
2. **Mixed time-frequency sampling matrix** $A_{\mathrm{mixed}}^{(d)}$ (configurable split between time and frequency)

for the integer-shifted sinc space $V_d = \mathrm{span}\{s_{n_0}, \dots, s_{n_0+d-1}\}$, where
$$s_n(t) = \operatorname{sinc}(t - n), \qquad n = n_0, n_0+1, \dots, n_0+d-1,$$
$\operatorname{sinc}(x) = \frac{\sin(\pi x)}{\pi x}$ with $\operatorname{sinc}(0) = 1$, and $n_0 = \texttt{SHIFT\_START}$ is a configurable integer.

### Fourier transform convention (unitary)
We use the unitary convention:
$$\hat{f}(\omega) = \frac{1}{\sqrt{2\pi}} \int_{-\infty}^{\infty} f(t)\, e^{-i\omega t}\, dt.$$
Under this convention:
$$\widehat{\operatorname{sinc}}(\omega) = \frac{1}{\sqrt{2\pi}}\, \mathbf{1}_{|\omega| \le \pi},$$
and an integer shift gives
$$\widehat{s_n}(\omega) = \frac{1}{\sqrt{2\pi}}\, e^{-i\omega n}\, \mathbf{1}_{|\omega| \le \pi}.$$

In [ ]:
import numpy as np
from scipy.linalg import svdvals
from scipy.sparse.linalg import svds
import matplotlib.pyplot as plt
import json
from pathlib import Path
import warnings
import os
warnings.filterwarnings('ignore')

# Create output folder for all results
os.makedirs('results/sinc_sampling_condition', exist_ok=True)

## Configuration Parameters

**`TIME_RATIO`**: Controls the fraction of samples allocated to the time domain in mixed sampling.
- `TIME_RATIO = 0.5` means $k = \lceil d/2 \rceil$ time samples, $m = d - k$ frequency samples
- `TIME_RATIO = 1.0` means all time samples (same as pure time)

**`TIME_INTERVAL`**: $[a, b]$ for time-domain sampling.

**`FREQ_INTERVAL`**: $[a, b]$ for frequency-domain sampling. The sinc Fourier transform is supported on $|\omega| \le \pi$, so frequency samples outside this range will see zero.

**`SHIFT_START`**: Integer $n_0$ controlling the first sinc shift. The basis is $\{s_{n_0}, s_{n_0+1}, \dots, s_{n_0+d-1}\}$. For example, `SHIFT_START = 0` gives shifts $0, 1, \dots, d-1$; `SHIFT_START = -5` gives shifts $-5, -4, \dots, d-6$.

In [ ]:
# =============================================================================
# CONFIGURATION: Sampling parameters
# =============================================================================

# Time/Frequency ratio for mixed sampling
TIME_RATIO = 0.5 # Fraction of samples in time domain (0 to 1)
                  # 0.5 = half time, half freq (ceil for time if odd)
                  # 1.0 = all time (same as pure time)
                  # 0.0 = all frequency

# Sampling intervals
TIME_INTERVAL = (-4.5, 4.5)  # [a, b] for time-domain samples
FREQ_INTERVAL = (-3.0, 3.0)  # [a, b] for frequency-domain samples

# Sinc shift range
SHIFT_START = 0  # Integer n_0: basis is sinc(t - n) for n = n_0, ..., n_0 + d - 1

# =============================================================================
# Validation and summary
# =============================================================================
assert 0.0 <= TIME_RATIO <= 1.0, "TIME_RATIO must be between 0 and 1"
assert TIME_INTERVAL[0] < TIME_INTERVAL[1], "TIME_INTERVAL must have a < b"
assert FREQ_INTERVAL[0] < FREQ_INTERVAL[1], "FREQ_INTERVAL must have a < b"
assert isinstance(SHIFT_START, int), "SHIFT_START must be an integer"

print("Configuration Summary")
print("="*60)
print(f"TIME_RATIO   = {TIME_RATIO}")
print(f"TIME_INTERVAL = [{TIME_INTERVAL[0]}, {TIME_INTERVAL[1]}]")
print(f"FREQ_INTERVAL = [{FREQ_INTERVAL[0]}, {FREQ_INTERVAL[1]}]")
print(f"SHIFT_START  = {SHIFT_START}")
print()
print("Example splits:")
for d in [10, 100, 500]:
    k = int(np.ceil(TIME_RATIO * d))
    m = d - k
    shifts = f"[{SHIFT_START}, ..., {SHIFT_START + d - 1}]"
    print(f"  d={d:4d}: k={k:4d} time, m={m:4d} freq, shifts {shifts}")

## 1. Define the sampling grids

For pure time sampling, use $d$ equispaced samples on `TIME_INTERVAL`.

For mixed sampling with configurable ratio:
- $k = \lceil \texttt{TIME\_RATIO} \cdot d \rceil$ time samples on `TIME_INTERVAL`
- $m = d - k$ frequency samples on `FREQ_INTERVAL`

In [ ]:
def build_grid(num_points, interval):
    """Build an equispaced grid on [a, b] with num_points points.
    
    Parameters:
        num_points: number of grid points
        interval: tuple (a, b) specifying the interval
    
    Returns:
        array of equispaced points
    """
    a, b = interval
    if num_points == 0:
        return np.array([])
    if num_points == 1:
        return np.array([(a + b) / 2.0])  # midpoint for single sample
    else:
        # j = 1, ..., num_points => indices 0, ..., num_points-1 in Python
        j = np.arange(1, num_points + 1)
        return a + (b - a) * (j - 1) / (num_points - 1)


def compute_time_freq_split(d, time_ratio):
    """Compute the number of time and frequency samples for given d and ratio.
    
    Parameters:
        d: total dimension
        time_ratio: fraction of samples in time domain (0 to 1)
    
    Returns:
        k: number of time samples (ceil of ratio * d)
        m: number of frequency samples (d - k)
    """
    k = int(np.ceil(time_ratio * d))
    # Ensure k is at least 0 and at most d
    k = max(0, min(d, k))
    m = d - k
    return k, m

## 2. Sinc basis evaluation

The basis functions are $s_n(t) = \operatorname{sinc}(t - n)$ for $n = n_0, \dots, n_0 + d - 1$, where $n_0 = \texttt{SHIFT\_START}$.

Given a grid of points, we build the evaluation matrix $B$ with $B_{j,n} = \operatorname{sinc}(t_j - (n_0 + n))$ for $n = 0, \dots, d-1$.

In [ ]:
def build_sinc_matrix(grid, d, shift_start=0):
    """Build r x d matrix B where B[:,n] = sinc(grid - (shift_start + n)) for n=0,...,d-1.
    
    Uses np.sinc which computes sinc(x) = sin(pi*x)/(pi*x).
    
    Parameters:
        grid: array of evaluation points (length r)
        d: number of basis functions
        shift_start: integer n_0, first shift index
    
    Returns:
        B: r x d matrix
    """
    r = len(grid)
    if r == 0:
        return np.empty((0, d))
    n_indices = np.arange(shift_start, shift_start + d)  # (d,)
    # B[j, n] = sinc(grid[j] - (shift_start + n))
    B = np.sinc(grid[:, None] - n_indices[None, :])  # (r, d)
    return B

## 3. Build the sampling matrices

### (A) Pure time sampling matrix
$$A_{\mathrm{time}}^{(d)} \in \mathbb{R}^{d \times d}, \quad (A_{\mathrm{time}}^{(d)})_{j,n} = \operatorname{sinc}(t_j^{(d)} - (n_0 + n))$$
where $t_j^{(d)}$ are $d$ equispaced points on `TIME_INTERVAL` and $n = 0, \dots, d-1$.

### (B) Mixed time-frequency sampling matrix
With $k = \lceil \texttt{TIME\_RATIO} \cdot d \rceil$ time samples and $m = d - k$ frequency samples:
$$A_{\mathrm{mixed}}^{(d)} = \begin{pmatrix} B_t \\ B_\omega \end{pmatrix}$$
where:
- $(B_t)_{j,n} = \operatorname{sinc}(t_j - (n_0 + n))$, with $k$ time samples on `TIME_INTERVAL`
- $(B_\omega)_{j,n} = \widehat{s_{n_0+n}}(\omega_j) = \frac{1}{\sqrt{2\pi}}\, e^{-i\omega_j (n_0+n)}\, \mathbf{1}_{|\omega_j| \le \pi}$, with $m$ frequency samples on `FREQ_INTERVAL`

The $1/\sqrt{2\pi}$ factor is a global row scaling that does not affect the condition number, but we include it for consistency with the unitary convention.

In [ ]:
def build_pure_time_matrix(d, time_interval, shift_start=0):
    """Build the d x d pure time sampling matrix A_time^(d).
    
    A^(d)_{j,n} = sinc(t^(d)_j - (shift_start + n)) for n=0,...,d-1.
    
    Parameters:
        d: dimension
        time_interval: tuple (a, b) for the sampling interval
        shift_start: integer n_0, first shift index
    """
    t = build_grid(d, time_interval)
    return build_sinc_matrix(t, d, shift_start)


def build_mixed_matrix(d, time_ratio, time_interval, freq_interval, shift_start=0):
    """Build the d x d mixed time-frequency sampling matrix A_mixed^(d).
    
    Uses k = ceil(time_ratio * d) time samples and m = d - k frequency samples.
    
    A_mixed = [B_t; B_omega]
    where:
      B_t[j, n] = sinc(t_j - (shift_start + n))
      B_omega[j, n] = (1/sqrt(2*pi)) * sinc_hat(omega_j) * exp(-i * omega_j * (shift_start + n))
    
    Under the unitary FT convention, sinc_hat(omega) = 1{|omega| <= pi}.
    
    Parameters:
        d: dimension
        time_ratio: fraction of samples in time domain (0 to 1)
        time_interval: tuple (a, b) for time-domain sampling
        freq_interval: tuple (a, b) for frequency-domain sampling
        shift_start: integer n_0, first shift index
    
    Returns:
        A_mixed: d x d complex matrix
        k: number of time samples
        m: number of frequency samples
    """
    k, m = compute_time_freq_split(d, time_ratio)
    
    # Build time grid and frequency grid
    t_grid = build_grid(k, time_interval)
    omega_grid = build_grid(m, freq_interval)
    
    # Time rows: B_t[j, n] = sinc(t_j - (shift_start + n))
    B_t = build_sinc_matrix(t_grid, d, shift_start)  # k x d real matrix
    
    # Frequency rows (unitary FT convention):
    # B_omega[j, n] = (1/sqrt(2*pi)) * 1{|omega_j| <= pi} * exp(-i * omega_j * (shift_start + n))
    n_indices = np.arange(shift_start, shift_start + d)  # (d,)
    if m > 0:
        sinc_hat = (np.abs(omega_grid) <= np.pi).astype(float)  # (m,)
        B_omega = (1.0 / np.sqrt(2 * np.pi)) * sinc_hat[:, None] * \
                  np.exp(-1j * omega_grid[:, None] * n_indices[None, :])  # (m, d)
    else:
        B_omega = np.empty((0, d), dtype=np.complex128)
    
    # Stack time rows and frequency rows
    A_mixed = np.vstack([B_t, B_omega]).astype(np.complex128)
    
    return A_mixed, k, m

## 4. Compute condition number

Use full SVD for $d \leq 2000$, and iterative methods for larger $d$.

In [ ]:
def compute_condition_number(A, d, svd_cutoff=2000):
    """Compute the 2-norm condition number of matrix A.
    
    Uses full SVD for d <= svd_cutoff, iterative SVD for larger d.
    Returns inf if sigma_min underflows to zero.
    """
    if d <= svd_cutoff:
        # Full SVD - O(d^3) but accurate
        try:
            s = svdvals(A)
            sigma_max = s[0]
            sigma_min = s[-1]
            if sigma_min == 0 or not np.isfinite(sigma_min):
                return np.inf, sigma_max, 0.0
            return sigma_max / sigma_min, sigma_max, sigma_min
        except Exception:
            return np.inf, np.nan, np.nan
    else:
        # Iterative SVD for large matrices
        try:
            _, s_max, _ = svds(A, k=1, which='LM', return_singular_vectors=True)
            sigma_max = s_max[0]
            _, s_min, _ = svds(A, k=1, which='SM', return_singular_vectors=True)
            sigma_min = s_min[0]
            if sigma_min == 0 or not np.isfinite(sigma_min):
                return np.inf, sigma_max, 0.0
            return sigma_max / sigma_min, sigma_max, sigma_min
        except Exception:
            return np.inf, np.nan, np.nan

## 5. Define the set of dimensions to test

Using $d = 1, \dots, 30$.

In [ ]:
def get_test_dimensions():
    """Generate the set of dimensions to test."""
    dims = set()
    
    # All d = 1, ..., 30
    dims.update(range(1, 31))
    
    return sorted(dims)

dimensions = get_test_dimensions()
print(f"Testing {len(dimensions)} dimensions from {min(dimensions)} to {max(dimensions)}")
print(f"First 20: {dimensions[:20]}")
print(f"Last 10: {dimensions[-10:]}")

## 6. Sanity checks for small dimensions

In [ ]:
print(f"Sanity checks for small dimensions (d <= 10):")
print(f"  TIME_RATIO   = {TIME_RATIO}")
print(f"  TIME_INTERVAL = {TIME_INTERVAL}")
print(f"  FREQ_INTERVAL = {FREQ_INTERVAL}")
print(f"  SHIFT_START  = {SHIFT_START}")
print("="*100)
print(f"{'d':>3} | {'k':>3} | {'m':>3} | {'k/d':>5} | {'shifts':>16} | k+m=d | Time finite | Mixed finite | Time kappa | Mixed kappa")
print("-"*120)

for d in range(1, 11):
    # Pure time matrix
    A_time = build_pure_time_matrix(d, TIME_INTERVAL, SHIFT_START)
    time_finite = np.all(np.isfinite(A_time))
    kappa_time, _, _ = compute_condition_number(A_time, d)
    
    # Mixed matrix
    A_mixed, k, m = build_mixed_matrix(d, TIME_RATIO, TIME_INTERVAL, FREQ_INTERVAL, SHIFT_START)
    mixed_finite = np.all(np.isfinite(A_mixed))
    kappa_mixed, _, _ = compute_condition_number(A_mixed, d)
    
    # Verify k + m = d
    km_check = "PASS" if k + m == d else "FAIL"
    k_ratio = k / d if d > 0 else 0
    shifts_str = f"[{SHIFT_START}..{SHIFT_START + d - 1}]"
    
    print(f"{d:3d} | {k:3d} | {m:3d} | {k_ratio:5.2f} | {shifts_str:>16} | {km_check:5s} | {str(time_finite):11s} | {str(mixed_finite):12s} | {kappa_time:10.2e} | {kappa_mixed:10.2e}")

print(f"\nNote: k/d should be approximately {TIME_RATIO} (with ceiling rounding).")

In [ ]:
# Verify grids and frequency-domain entries for d=4
print(f"Verification of grids and frequency entries for d=4:")
print(f"  TIME_RATIO={TIME_RATIO}, TIME_INTERVAL={TIME_INTERVAL}, FREQ_INTERVAL={FREQ_INTERVAL}")
print(f"  SHIFT_START={SHIFT_START}")
print("="*70)
d = 4
A_mixed, k, m = build_mixed_matrix(d, TIME_RATIO, TIME_INTERVAL, FREQ_INTERVAL, SHIFT_START)
print(f"k={k} time rows, m={m} frequency rows")
print(f"Shifts: n = {SHIFT_START}, {SHIFT_START+1}, ..., {SHIFT_START+d-1}")

# Show grids
t_grid = build_grid(k, TIME_INTERVAL)
omega_grid = build_grid(m, FREQ_INTERVAL)
print(f"\nTime grid ({k} points on {TIME_INTERVAL}): {t_grid}")
print(f"Freq grid ({m} points on {FREQ_INTERVAL}): {omega_grid}")

print(f"\nsinc_hat(omega_j) = (1/sqrt(2*pi)) * 1{{|omega_j| <= pi}} for each freq sample:")
for j, w in enumerate(omega_grid):
    in_band = abs(w) <= np.pi
    val = 1.0 / np.sqrt(2 * np.pi) if in_band else 0.0
    print(f"  omega_{j} = {w:.4f}: |omega| <= pi? {in_band}  =>  sinc_hat = {val:.6f}")

print(f"\nMixed matrix A_mixed (shape {A_mixed.shape}):")
print(f"Time rows (first {k} rows, real part):")
print(A_mixed[:k, :].real)
if m > 0:
    print(f"\nFrequency rows (last {m} rows):")
    print(A_mixed[k:, :])
else:
    print("\nNo frequency rows (all samples in time domain).")

## 7. Compute condition numbers for all dimensions

In [ ]:
results = []
output_file = Path("results/sinc_sampling_condition/sinc_condition_numbers.json")

print(f"Computing condition numbers...")
print(f"  TIME_RATIO={TIME_RATIO}, TIME_INTERVAL={TIME_INTERVAL}, FREQ_INTERVAL={FREQ_INTERVAL}")
print(f"  SHIFT_START={SHIFT_START}")
print("="*85)

for i, d in enumerate(dimensions):
    # Build pure time sampling matrix
    A_time = build_pure_time_matrix(d, TIME_INTERVAL, SHIFT_START)
    kappa_time, sigma_max_time, sigma_min_time = compute_condition_number(A_time, d)
    
    # Build mixed sampling matrix
    A_mixed, k, m = build_mixed_matrix(d, TIME_RATIO, TIME_INTERVAL, FREQ_INTERVAL, SHIFT_START)
    kappa_mixed, sigma_max_mixed, sigma_min_mixed = compute_condition_number(A_mixed, d)
    
    # Store results
    result = {
        "d": d,
        "time_ratio": TIME_RATIO,
        "time_interval": list(TIME_INTERVAL),
        "freq_interval": list(FREQ_INTERVAL),
        "shift_start": SHIFT_START,
        "k": k,
        "m": m,
        "kappa_time": float(kappa_time) if np.isfinite(kappa_time) else "inf",
        "kappa_mixed": float(kappa_mixed) if np.isfinite(kappa_mixed) else "inf",
        "sigma_max_time": float(sigma_max_time) if np.isfinite(sigma_max_time) else "nan",
        "sigma_min_time": float(sigma_min_time) if np.isfinite(sigma_min_time) else "nan",
        "sigma_max_mixed": float(sigma_max_mixed) if np.isfinite(sigma_max_mixed) else "nan",
        "sigma_min_mixed": float(sigma_min_mixed) if np.isfinite(sigma_min_mixed) else "nan",
        "svd_method": "full" if d <= 2000 else "iterative"
    }
    results.append(result)
    
    # Progress update
    if (i + 1) % 20 == 0 or d in [1, 10, 100, 500]:
        kt_str = f"{kappa_time:.4e}" if np.isfinite(kappa_time) else "inf"
        km_str = f"{kappa_mixed:.4e}" if np.isfinite(kappa_mixed) else "inf"
        print(f"[{i+1:3d}/{len(dimensions)}] d={d:5d} (k={k}, m={m}): kappa_time = {kt_str}, kappa_mixed = {km_str}")
    
    # Save incrementally every 50 iterations
    if (i + 1) % 50 == 0:
        with open(output_file, 'w') as f:
            json.dump(results, f, indent=2)

# Final save
with open(output_file, 'w') as f:
    json.dump(results, f, indent=2)

print("\nComputation complete!")
print(f"Results saved to {output_file}")

## 8. Save as CSV

In [ ]:
csv_file = Path("results/sinc_sampling_condition/sinc_condition_numbers.csv")

with open(csv_file, 'w') as f:
    f.write("d,time_ratio,time_interval_a,time_interval_b,freq_interval_a,freq_interval_b,k,m,kappa_time,kappa_mixed,sigma_max_time,sigma_min_time,sigma_max_mixed,sigma_min_mixed\n")
    for r in results:
        f.write(f"{r['d']},{r['time_ratio']},{r['time_interval'][0]},{r['time_interval'][1]},{r['freq_interval'][0]},{r['freq_interval'][1]},{r['k']},{r['m']},{r['kappa_time']},{r['kappa_mixed']},{r['sigma_max_time']},{r['sigma_min_time']},{r['sigma_max_mixed']},{r['sigma_min_mixed']}\n")

print(f"Results saved to {csv_file}")

## 9. Plot results: Pure Time vs Mixed Sampling

In [ ]:
# Extract data for plotting
d_vals = np.array([r['d'] for r in results])
kappa_time_vals = np.array([r['kappa_time'] if r['kappa_time'] != 'inf' else np.inf for r in results])
kappa_mixed_vals = np.array([r['kappa_mixed'] if r['kappa_mixed'] != 'inf' else np.inf for r in results])

# Filter out infinite values for plotting
time_finite_mask = np.isfinite(kappa_time_vals)
mixed_finite_mask = np.isfinite(kappa_mixed_vals)

d_time_finite = d_vals[time_finite_mask]
kappa_time_finite = kappa_time_vals[time_finite_mask]

d_mixed_finite = d_vals[mixed_finite_mask]
kappa_mixed_finite = kappa_mixed_vals[mixed_finite_mask]

print(f"Pure time: {len(d_time_finite)} finite out of {len(d_vals)}")
print(f"Mixed (TIME_RATIO={TIME_RATIO}): {len(d_mixed_finite)} finite out of {len(d_vals)}")

# Find where condition numbers become infinite
if np.any(~time_finite_mask):
    first_inf_time = d_vals[~time_finite_mask][0]
    print(f"Pure time becomes numerically infinite at d = {first_inf_time}")
else:
    first_inf_time = None
    
if np.any(~mixed_finite_mask):
    first_inf_mixed = d_vals[~mixed_finite_mask][0]
    print(f"Mixed becomes numerically infinite at d = {first_inf_mixed}")
else:
    first_inf_mixed = None

In [ ]:
# Create labels for sampling methods
time_label = f'Pure time ({TIME_INTERVAL})'

if TIME_RATIO == 0.5:
    mixed_label = f'Mixed (1/2 time {TIME_INTERVAL} + 1/2 freq {FREQ_INTERVAL})'
else:
    from fractions import Fraction
    frac = Fraction(TIME_RATIO).limit_denominator(100)
    if frac.denominator <= 10:
        mixed_label = f'Mixed ({frac.numerator}/{frac.denominator} time {TIME_INTERVAL} + {frac.denominator - frac.numerator}/{frac.denominator} freq {FREQ_INTERVAL})'
    else:
        mixed_label = f'Mixed ({TIME_RATIO:.2f} time {TIME_INTERVAL} + {1-TIME_RATIO:.2f} freq {FREQ_INTERVAL})'

fig, ax1 = plt.subplots(1, 1, figsize=(10, 6))

# Plot: Linear x-axis, log y-axis
ax1.semilogy(d_time_finite, kappa_time_finite, 'b.-', markersize=3, linewidth=0.8, 
             label=time_label)
ax1.semilogy(d_mixed_finite, kappa_mixed_finite, 'r.-', markersize=3, linewidth=0.8,
             label=mixed_label)
ax1.set_xlabel('Dimension d', fontsize=12)
ax1.set_ylabel(r'$\kappa_2(A^{(d)})$', fontsize=12)
ax1.set_title(f'Sinc Basis: Condition Number — Pure Time vs Mixed', fontsize=13)
ax1.grid(True, which='both', alpha=0.3)
ax1.legend(loc='upper left', fontsize=9)
ax1.set_xlim([0, max(d_vals) * 1.02])

plt.tight_layout()
plt.savefig('results/sinc_sampling_condition/sinc_condition_number_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nPlot saved to results/sinc_sampling_condition/sinc_condition_number_comparison.png")

## 10. Summary statistics

In [ ]:
print("Summary Statistics")
print("="*75)
print(f"Configuration:")
print(f"  TIME_RATIO   = {TIME_RATIO}")
print(f"  TIME_INTERVAL = {TIME_INTERVAL}")
print(f"  FREQ_INTERVAL = {FREQ_INTERVAL}")
print(f"  SHIFT_START  = {SHIFT_START}")
print(f"Total dimensions tested: {len(d_vals)}")

print(f"\nPure Time Sampling (on {TIME_INTERVAL}):")
print(f"  Finite condition numbers: {len(d_time_finite)}")
if len(kappa_time_finite) > 0:
    print(f"  Min kappa: {np.min(kappa_time_finite):.4e} at d = {d_time_finite[np.argmin(kappa_time_finite)]}")
    print(f"  Max kappa: {np.max(kappa_time_finite):.4e} at d = {d_time_finite[np.argmax(kappa_time_finite)]}")

print(f"\nMixed Sampling (time on {TIME_INTERVAL}, freq on {FREQ_INTERVAL}):")
print(f"  Finite condition numbers: {len(d_mixed_finite)}")
if len(kappa_mixed_finite) > 0:
    print(f"  Min kappa: {np.min(kappa_mixed_finite):.4e} at d = {d_mixed_finite[np.argmin(kappa_mixed_finite)]}")
    print(f"  Max kappa: {np.max(kappa_mixed_finite):.4e} at d = {d_mixed_finite[np.argmax(kappa_mixed_finite)]}")

print(f"\nCondition numbers at key dimensions:")
print(f"{'d':>6} | {'k':>5} | {'m':>5} | {'kappa_time':>14} | {'kappa_mixed':>14} | {'Ratio (time/mixed)':>18}")
print("-"*80)
key_dims = [1, 2, 5, 10, 20, 30]
for kd in key_dims:
    idx = np.where(d_vals == kd)[0]
    if len(idx) > 0:
        r = results[idx[0]]
        kt = kappa_time_vals[idx[0]]
        km = kappa_mixed_vals[idx[0]]
        kt_str = f"{kt:.4e}" if np.isfinite(kt) else "inf"
        km_str = f"{km:.4e}" if np.isfinite(km) else "inf"
        if np.isfinite(kt) and np.isfinite(km) and km > 0:
            ratio = kt / km
            ratio_str = f"{ratio:.2e}"
        else:
            ratio_str = "N/A"
        print(f"{kd:6d} | {r['k']:5d} | {r['m']:5d} | {kt_str:>14} | {km_str:>14} | {ratio_str:>18}")

## 11. Ratio analysis: How much better is mixed sampling?

In [ ]:
# Compute ratio where both are finite
both_finite = time_finite_mask & mixed_finite_mask
d_both = d_vals[both_finite]
ratio = kappa_time_vals[both_finite] / kappa_mixed_vals[both_finite]

fig, ax = plt.subplots(figsize=(10, 5))
ax.semilogy(d_both, ratio, 'g.-', markersize=3, linewidth=0.8)
ax.axhline(y=1, color='k', linestyle='--', alpha=0.5, label='Ratio = 1 (equal)')
ax.set_xlabel('Dimension d', fontsize=12)
ax.set_ylabel(r'$\kappa_2(A_{\mathrm{time}}) / \kappa_2(A_{\mathrm{mixed}})$', fontsize=12)
ax.set_title(f'Sinc Basis: Condition Number Ratio — Pure Time / Mixed\n(TIME_RATIO={TIME_RATIO}, time={TIME_INTERVAL}, freq={FREQ_INTERVAL})', fontsize=12)
ax.grid(True, which='both', alpha=0.3)
ax.legend()

plt.tight_layout()
plt.savefig('results/sinc_sampling_condition/sinc_condition_ratio.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nRatio statistics (pure time / mixed):")
print(f"  Configuration: TIME_RATIO={TIME_RATIO}, TIME_INTERVAL={TIME_INTERVAL}, FREQ_INTERVAL={FREQ_INTERVAL}")
print(f"  Min ratio: {np.min(ratio):.4e} at d = {d_both[np.argmin(ratio)]}")
print(f"  Max ratio: {np.max(ratio):.4e} at d = {d_both[np.argmax(ratio)]}")
print(f"  Mean ratio: {np.mean(ratio):.4e}")
print(f"  Median ratio: {np.median(ratio):.4e}")

# =============================================================================
# EXPERIMENT 1b: Regular Sampling with d-Dependent Shifts and Time Interval
# =============================================================================

The sinc shifts and time interval now scale with $d$:
- **Shifts**: $n_0(d) = \lceil -d/2+1 \rceil$, so the shifts are $n_0, n_0+1, \dots, n_0+d-1$ (roughly centred around 0).
- **Time interval**: $[\lceil -d/2+1 \rceil,\; \lfloor d/2 \rfloor]$, matching the support of the shifted sincs.
- **Frequency interval**: unchanged (same `FREQ_INTERVAL` as configured above).

In [ ]:
# =============================================================================
# EXPERIMENT 1b: Regular Sampling with d-Dependent Shifts and Time Interval
# =============================================================================

results_ddep = []
output_file_ddep = Path("results/sinc_sampling_condition/sinc_condition_numbers_ddep.json")

print("Computing condition numbers with d-DEPENDENT shifts and time interval...")
print(f"  Shifts: n_0(d) = ceil(-d/2+1), time interval = [ceil(-d/2+1), floor(d/2)]")
print(f"  FREQ_INTERVAL = {FREQ_INTERVAL} (fixed)")
print(f"  TIME_RATIO = {TIME_RATIO}")
print("="*90)

for i, d in enumerate(dimensions):
    # d-dependent shift start and time interval
    shift_start_d = int(np.ceil(-d / 2 + 1))
    shift_end_d = shift_start_d + d - 1  # = floor(d/2)
    time_interval_d = (float(shift_start_d), float(shift_end_d))

    # Build matrices with d-dependent shifts and time interval
    A_time = build_pure_time_matrix(d, time_interval_d, shift_start_d)
    kappa_time, sigma_max_time, sigma_min_time = compute_condition_number(A_time, d)

    A_mixed, k, m = build_mixed_matrix(d, TIME_RATIO, time_interval_d, FREQ_INTERVAL, shift_start_d)
    kappa_mixed, sigma_max_mixed, sigma_min_mixed = compute_condition_number(A_mixed, d)

    result = {
        "d": d,
        "shift_start": shift_start_d,
        "shift_end": shift_end_d,
        "time_interval": list(time_interval_d),
        "freq_interval": list(FREQ_INTERVAL),
        "time_ratio": TIME_RATIO,
        "k": k,
        "m": m,
        "kappa_time": float(kappa_time) if np.isfinite(kappa_time) else "inf",
        "kappa_mixed": float(kappa_mixed) if np.isfinite(kappa_mixed) else "inf",
        "sigma_max_time": float(sigma_max_time) if np.isfinite(sigma_max_time) else "nan",
        "sigma_min_time": float(sigma_min_time) if np.isfinite(sigma_min_time) else "nan",
        "sigma_max_mixed": float(sigma_max_mixed) if np.isfinite(sigma_max_mixed) else "nan",
        "sigma_min_mixed": float(sigma_min_mixed) if np.isfinite(sigma_min_mixed) else "nan",
    }
    results_ddep.append(result)

    if (i + 1) % 20 == 0 or d in [1, 10, 100, 500]:
        kt_str = f"{kappa_time:.4e}" if np.isfinite(kappa_time) else "inf"
        km_str = f"{kappa_mixed:.4e}" if np.isfinite(kappa_mixed) else "inf"
        print(f"[{i+1:3d}/{len(dimensions)}] d={d:5d}, shifts=[{shift_start_d}..{shift_end_d}], "
              f"time_int={time_interval_d}: kappa_time={kt_str}, kappa_mixed={km_str}")

# Final save
with open(output_file_ddep, 'w') as f:
    json.dump(results_ddep, f, indent=2)

print("="*90)
print(f"Saved results to {output_file_ddep}")

In [ ]:
# Extract d-dependent data for plotting
d_vals_ddep = np.array([r['d'] for r in results_ddep])
kappa_time_ddep = np.array([r['kappa_time'] if r['kappa_time'] != 'inf' else np.inf for r in results_ddep])
kappa_mixed_ddep = np.array([r['kappa_mixed'] if r['kappa_mixed'] != 'inf' else np.inf for r in results_ddep])

# Filter finite values
time_finite_mask_ddep = np.isfinite(kappa_time_ddep)
mixed_finite_mask_ddep = np.isfinite(kappa_mixed_ddep)

d_time_finite_ddep = d_vals_ddep[time_finite_mask_ddep]
kappa_time_finite_ddep = kappa_time_ddep[time_finite_mask_ddep]
d_mixed_finite_ddep = d_vals_ddep[mixed_finite_mask_ddep]
kappa_mixed_finite_ddep = kappa_mixed_ddep[mixed_finite_mask_ddep]

print(f"d-dependent: Pure time: {len(d_time_finite_ddep)} finite, Mixed: {len(d_mixed_finite_ddep)} finite out of {len(d_vals_ddep)}")

# Plot
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

ax.semilogy(d_time_finite_ddep, kappa_time_finite_ddep, 'b.-', linewidth=0.8, markersize=3,
            label='Pure Time (d-dep intervals)')
ax.semilogy(d_mixed_finite_ddep, kappa_mixed_finite_ddep, 'r.-', linewidth=0.8, markersize=3,
            label='Mixed (d-dep intervals)')

ax.set_xlabel('Dimension d', fontsize=12)
ax.set_ylabel(r'$\kappa_2(A^{(d)})$', fontsize=12)
ax.set_title(r'EXPERIMENT 1b: d-Dependent Shifts $[\lceil -d/2+1 \rceil \ldots \lfloor d/2 \rfloor]$'
             f'\nfreq interval = {FREQ_INTERVAL}', fontsize=13)
ax.legend(loc='upper left', fontsize=10)
ax.grid(True, which='both', alpha=0.3)
ax.set_xlim([0, max(dimensions) * 1.02])

plt.tight_layout()
plt.savefig('results/sinc_sampling_condition/sinc_condition_number_ddep_regular.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Saved plot to results/sinc_sampling_condition/sinc_condition_number_ddep_regular.png")

# Summary table
print(f"\nExperiment 1b: d-dependent shifts summary:")
print(f"{'d':>6} | {'n_0':>5} | {'n_end':>5} | {'time_int':>18} | {'kappa_time':>14} | {'kappa_mixed':>14}")
print("-"*80)
for kd in [1, 2, 5, 10, 15, 20, 25, 30]:
    idx = np.where(d_vals_ddep == kd)[0]
    if len(idx) > 0:
        r = results_ddep[idx[0]]
        kt = kappa_time_ddep[idx[0]]
        km = kappa_mixed_ddep[idx[0]]
        kt_str = f"{kt:.4e}" if np.isfinite(kt) else "inf"
        km_str = f"{km:.4e}" if np.isfinite(km) else "inf"
        print(f"{kd:6d} | {r['shift_start']:5d} | {r['shift_end']:5d} | {str(r['time_interval']):>18} | {kt_str:>14} | {km_str:>14}")

# =============================================================================
# EXPERIMENT 2: Random Sampling (Uniform in Intervals)
# =============================================================================
# This section compares pure time vs mixed sampling with RANDOM sample locations
# (uniformly distributed in the intervals) instead of regular grids.
# The randomness between pure and mixed sampling cases is INDEPENDENT.
# Results are AVERAGED over NUM_RANDOM_TRIALS trials for robustness.

In [ ]:
# =============================================================================
# CONFIGURATION: Random Sampling Parameters
# =============================================================================

NUM_RANDOM_TRIALS = 1000  # Number of random trials to average over
RANDOM_SEED = 42        # Base random seed for reproducibility

print(f"Random Sampling Configuration:")
print(f"  NUM_RANDOM_TRIALS = {NUM_RANDOM_TRIALS}")
print(f"  RANDOM_SEED = {RANDOM_SEED}")
print(f"  (Using same TIME_RATIO={TIME_RATIO}, TIME_INTERVAL={TIME_INTERVAL}, FREQ_INTERVAL={FREQ_INTERVAL})")
print(f"  SHIFT_START = {SHIFT_START}")

In [ ]:
def build_random_grid(num_points, interval, rng=None):
    """Build a random (uniform) grid on [a, b] with num_points points.
    
    Parameters:
        num_points: number of grid points
        interval: tuple (a, b) specifying the interval
        rng: numpy random generator (for reproducibility)
    
    Returns:
        array of uniformly distributed random points in [a, b]
    """
    a, b = interval
    if num_points == 0:
        return np.array([])
    if rng is None:
        rng = np.random.default_rng()
    return rng.uniform(a, b, size=num_points)


def build_pure_time_matrix_random(d, time_interval, shift_start=0, rng=None):
    """Build the d x d pure time sampling matrix with RANDOM sample locations.
    
    Parameters:
        d: dimension
        time_interval: tuple (a, b) for the sampling interval
        shift_start: integer n_0, first shift index
        rng: numpy random generator
    """
    t = build_random_grid(d, time_interval, rng)
    return build_sinc_matrix(t, d, shift_start)


def build_mixed_matrix_random(d, time_ratio, time_interval, freq_interval,
                               shift_start=0, rng_time=None, rng_freq=None):
    """Build the d x d mixed sampling matrix with RANDOM sample locations.
    
    Uses k = ceil(time_ratio * d) time samples and m = d - k frequency samples.
    Time and frequency samples are drawn independently from uniform distributions.
    
    Parameters:
        d: dimension
        time_ratio: fraction of samples in time domain (0 to 1)
        time_interval: tuple (a, b) for time-domain sampling
        freq_interval: tuple (a, b) for frequency-domain sampling
        shift_start: integer n_0, first shift index
        rng_time: random generator for time samples
        rng_freq: random generator for frequency samples
    
    Returns:
        A_mixed: d x d complex matrix
        k: number of time samples
        m: number of frequency samples
    """
    k, m = compute_time_freq_split(d, time_ratio)
    
    # Build random time grid and frequency grid
    t_grid = build_random_grid(k, time_interval, rng_time)
    omega_grid = build_random_grid(m, freq_interval, rng_freq)
    
    # Time rows: sinc evaluations
    B_t = build_sinc_matrix(t_grid, d, shift_start)  # k x d real matrix
    
    # Frequency rows (unitary FT convention)
    n_indices = np.arange(shift_start, shift_start + d)  # (d,)
    if m > 0:
        sinc_hat = (np.abs(omega_grid) <= np.pi).astype(float)  # (m,)
        B_omega = (1.0 / np.sqrt(2 * np.pi)) * sinc_hat[:, None] * \
                  np.exp(-1j * omega_grid[:, None] * n_indices[None, :])  # (m, d)
    else:
        B_omega = np.empty((0, d), dtype=np.complex128)
    
    # Stack time rows and frequency rows
    A_mixed = np.vstack([B_t, B_omega]).astype(np.complex128)
    
    return A_mixed, k, m


print("Random sampling functions defined.")

In [ ]:
results_random = []
output_file_random = Path("results/sinc_sampling_condition/sinc_condition_numbers_random.json")

print(f"Computing condition numbers with RANDOM sampling...")
print(f"  Averaging over {NUM_RANDOM_TRIALS} trials (seed={RANDOM_SEED})")
print(f"  TIME_RATIO={TIME_RATIO}, TIME_INTERVAL={TIME_INTERVAL}, FREQ_INTERVAL={FREQ_INTERVAL}")
print(f"  SHIFT_START={SHIFT_START}")
print("="*90)

for i, d in enumerate(dimensions):
    k, m = compute_time_freq_split(d, TIME_RATIO)
    
    # Collect condition numbers across all trials
    kappa_time_trials = []
    kappa_mixed_trials = []
    
    for trial in range(NUM_RANDOM_TRIALS):
        # Create independent random generators for pure time and mixed sampling
        trial_offset = trial * 100000
        
        # Pure time gets its own generator
        rng_pure = np.random.default_rng(RANDOM_SEED + d + trial_offset)
        
        # Mixed sampling gets separate generators for time and freq (independent from pure)
        rng_mixed_time = np.random.default_rng(RANDOM_SEED + d + trial_offset + 10000)
        rng_mixed_freq = np.random.default_rng(RANDOM_SEED + d + trial_offset + 20000)
        
        # Build pure time sampling matrix with random samples
        A_time_rand = build_pure_time_matrix_random(d, TIME_INTERVAL, SHIFT_START, rng_pure)
        kappa_time_rand, _, _ = compute_condition_number(A_time_rand, d)
        
        # Build mixed sampling matrix with random samples (independent randomness)
        A_mixed_rand, _, _ = build_mixed_matrix_random(d, TIME_RATIO, TIME_INTERVAL, FREQ_INTERVAL,
                                                        SHIFT_START, rng_mixed_time, rng_mixed_freq)
        kappa_mixed_rand, _, _ = compute_condition_number(A_mixed_rand, d)
        
        kappa_time_trials.append(kappa_time_rand if np.isfinite(kappa_time_rand) else np.inf)
        kappa_mixed_trials.append(kappa_mixed_rand if np.isfinite(kappa_mixed_rand) else np.inf)
    
    # Convert to arrays
    kappa_time_trials = np.array(kappa_time_trials)
    kappa_mixed_trials = np.array(kappa_mixed_trials)
    
    # Compute statistics over finite values
    time_finite_trials = kappa_time_trials[np.isfinite(kappa_time_trials)]
    mixed_finite_trials = kappa_mixed_trials[np.isfinite(kappa_mixed_trials)]
    
    kappa_time_mean = np.mean(time_finite_trials) if len(time_finite_trials) > 0 else np.inf
    kappa_mixed_mean = np.mean(mixed_finite_trials) if len(mixed_finite_trials) > 0 else np.inf
    
    kappa_time_std = np.std(time_finite_trials) if len(time_finite_trials) > 1 else 0.0
    kappa_mixed_std = np.std(mixed_finite_trials) if len(mixed_finite_trials) > 1 else 0.0
    
    kappa_time_min = np.min(time_finite_trials) if len(time_finite_trials) > 0 else np.inf
    kappa_time_max = np.max(time_finite_trials) if len(time_finite_trials) > 0 else np.inf
    kappa_mixed_min = np.min(mixed_finite_trials) if len(mixed_finite_trials) > 0 else np.inf
    kappa_mixed_max = np.max(mixed_finite_trials) if len(mixed_finite_trials) > 0 else np.inf
    
    result = {
        "d": d,
        "time_ratio": TIME_RATIO,
        "time_interval": list(TIME_INTERVAL),
        "freq_interval": list(FREQ_INTERVAL),
        "shift_start": SHIFT_START,
        "k": k,
        "m": m,
        "num_trials": NUM_RANDOM_TRIALS,
        "kappa_time_mean": float(kappa_time_mean) if np.isfinite(kappa_time_mean) else "inf",
        "kappa_time_std": float(kappa_time_std) if np.isfinite(kappa_time_std) else "nan",
        "kappa_time_min": float(kappa_time_min) if np.isfinite(kappa_time_min) else "inf",
        "kappa_time_max": float(kappa_time_max) if np.isfinite(kappa_time_max) else "inf",
        "kappa_mixed_mean": float(kappa_mixed_mean) if np.isfinite(kappa_mixed_mean) else "inf",
        "kappa_mixed_std": float(kappa_mixed_std) if np.isfinite(kappa_mixed_std) else "nan",
        "kappa_mixed_min": float(kappa_mixed_min) if np.isfinite(kappa_mixed_min) else "inf",
        "kappa_mixed_max": float(kappa_mixed_max) if np.isfinite(kappa_mixed_max) else "inf",
        "num_finite_time_trials": len(time_finite_trials),
        "num_finite_mixed_trials": len(mixed_finite_trials),
        "random_seed": RANDOM_SEED
    }
    results_random.append(result)
    
    # Progress update
    if (i + 1) % 20 == 0 or d in [1, 10, 100, 500]:
        kt_str = f"{kappa_time_mean:.4e}" if np.isfinite(kappa_time_mean) else "inf"
        km_str = f"{kappa_mixed_mean:.4e}" if np.isfinite(kappa_mixed_mean) else "inf"
        print(f"[{i+1:3d}/{len(dimensions)}] d={d:5d} (k={k}, m={m}): mean kappa_time = {kt_str}, mean kappa_mixed = {km_str}")
    
    # Save incrementally every 50 iterations
    if (i + 1) % 50 == 0:
        with open(output_file_random, 'w') as f:
            json.dump(results_random, f, indent=2)

# Final save
with open(output_file_random, 'w') as f:
    json.dump(results_random, f, indent=2)

print("\nComputation complete!")
print(f"Results saved to {output_file_random}")

In [ ]:
# Extract random sampling data for plotting
d_vals_rand = np.array([r['d'] for r in results_random])

kappa_time_rand_mean = np.array([
    r['kappa_time_mean'] if r['kappa_time_mean'] != 'inf' else np.inf 
    for r in results_random
])
kappa_mixed_rand_mean = np.array([
    r['kappa_mixed_mean'] if r['kappa_mixed_mean'] != 'inf' else np.inf
    for r in results_random
])

kappa_time_rand_std = np.array([
    r.get('kappa_time_std', 0.0) if r.get('kappa_time_std', 'nan') != 'nan' else 0.0
    for r in results_random
])
kappa_mixed_rand_std = np.array([
    r.get('kappa_mixed_std', 0.0) if r.get('kappa_mixed_std', 'nan') != 'nan' else 0.0
    for r in results_random
])

# Filter out infinite values for plotting
time_rand_finite_mask = np.isfinite(kappa_time_rand_mean)
mixed_rand_finite_mask = np.isfinite(kappa_mixed_rand_mean)

d_time_rand_finite = d_vals_rand[time_rand_finite_mask]
kappa_time_rand_finite = kappa_time_rand_mean[time_rand_finite_mask]
kappa_time_rand_std_finite = kappa_time_rand_std[time_rand_finite_mask]

d_mixed_rand_finite = d_vals_rand[mixed_rand_finite_mask]
kappa_mixed_rand_finite = kappa_mixed_rand_mean[mixed_rand_finite_mask]
kappa_mixed_rand_std_finite = kappa_mixed_rand_std[mixed_rand_finite_mask]

num_trials_used = results_random[0].get('num_trials', 1) if results_random else 1

print(f"Random Sampling Results (averaged over {num_trials_used} trials):")
print(f"  Pure time (random): {len(d_time_rand_finite)} finite out of {len(d_vals_rand)}")
print(f"  Mixed (random): {len(d_mixed_rand_finite)} finite out of {len(d_vals_rand)}")

### Random Sampling: Pure Time vs Mixed (standalone)

In [ ]:
# Plot: Random Sampling - Pure Time vs Mixed (with error bands)
fig, ax1 = plt.subplots(1, 1, figsize=(10, 6))

time_rand_label = f'Pure time RANDOM (mean +/- std, n={num_trials_used})'
mixed_rand_label = f'Mixed RANDOM (mean +/- std, n={num_trials_used})'

ax1.semilogy(d_time_rand_finite, kappa_time_rand_finite, 'b.-', markersize=3, linewidth=0.8, 
             label=time_rand_label)
if np.any(kappa_time_rand_std_finite > 0):
    ax1.fill_between(d_time_rand_finite, 
                     np.maximum(kappa_time_rand_finite - kappa_time_rand_std_finite, 1e-10),
                     kappa_time_rand_finite + kappa_time_rand_std_finite,
                     alpha=0.2, color='blue')
ax1.semilogy(d_mixed_rand_finite, kappa_mixed_rand_finite, 'r.-', markersize=3, linewidth=0.8,
             label=mixed_rand_label)
if np.any(kappa_mixed_rand_std_finite > 0):
    ax1.fill_between(d_mixed_rand_finite,
                     np.maximum(kappa_mixed_rand_finite - kappa_mixed_rand_std_finite, 1e-10),
                     kappa_mixed_rand_finite + kappa_mixed_rand_std_finite,
                     alpha=0.2, color='red')
ax1.set_xlabel('Dimension d', fontsize=12)
ax1.set_ylabel(r'$\kappa_2(A^{(d)})$', fontsize=12)
ax1.set_title(f'Sinc Basis: Random Sampling (avg over {num_trials_used} trials)\nPure Time vs Mixed', fontsize=13)
ax1.grid(True, which='both', alpha=0.3)
ax1.legend(loc='upper left', fontsize=9)
ax1.set_xlim([0, max(d_vals_rand) * 1.02])

plt.tight_layout()
plt.savefig('results/sinc_sampling_condition/sinc_condition_number_random.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nPlot saved to results/sinc_sampling_condition/sinc_condition_number_random.png")
print(f"(Shaded regions show +/-1 standard deviation from {num_trials_used} trials)")

### Comparison: Regular vs Random Sampling (all 4 curves)

In [ ]:
# Plot all 4 curves: Regular and Random (mean) for both Pure Time and Mixed
fig, ax1 = plt.subplots(1, 1, figsize=(10, 6))

rand_label_suffix = f' (mean, n={num_trials_used})'

# Regular sampling
ax1.semilogy(d_time_finite, kappa_time_finite, 'b-', markersize=2, linewidth=1.2, 
             label=f'Pure time REGULAR')
ax1.semilogy(d_mixed_finite, kappa_mixed_finite, 'r-', markersize=2, linewidth=1.2,
             label=f'Mixed REGULAR')
# Random sampling (mean)
ax1.semilogy(d_time_rand_finite, kappa_time_rand_finite, 'c--', markersize=2, linewidth=1.2, 
             label=f'Pure time RANDOM{rand_label_suffix}')
ax1.semilogy(d_mixed_rand_finite, kappa_mixed_rand_finite, 'm--', markersize=2, linewidth=1.2,
             label=f'Mixed RANDOM{rand_label_suffix}')

ax1.set_xlabel('Dimension d', fontsize=12)
ax1.set_ylabel(r'$\kappa_2(A^{(d)})$', fontsize=12)
ax1.set_title(f'Sinc Basis: Condition Number — Regular vs Random Sampling\nTIME_RATIO={TIME_RATIO}, time={TIME_INTERVAL}, freq={FREQ_INTERVAL}, shifts=[{SHIFT_START}..{SHIFT_START}+d-1]', fontsize=11)
ax1.grid(True, which='both', alpha=0.3)
ax1.legend(loc='upper left', fontsize=9)
ax1.set_xlim([0, max(dimensions) * 1.02])

plt.tight_layout()
plt.savefig('results/sinc_sampling_condition/sinc_condition_number_all_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nPlot saved to results/sinc_sampling_condition/sinc_condition_number_all_comparison.png")

### Random Sampling: Summary Statistics

In [ ]:
# Summary statistics for random sampling
print(f"Summary Statistics: RANDOM Sampling (averaged over {num_trials_used} trials)")
print("="*85)
print(f"Configuration:")
print(f"  TIME_RATIO   = {TIME_RATIO}")
print(f"  TIME_INTERVAL = {TIME_INTERVAL}")
print(f"  FREQ_INTERVAL = {FREQ_INTERVAL}")
print(f"  SHIFT_START  = {SHIFT_START}")
print(f"  NUM_RANDOM_TRIALS = {num_trials_used}")
print(f"  RANDOM_SEED = {results_random[0].get('random_seed', 'N/A')}")
print(f"Total dimensions tested: {len(d_vals_rand)}")

print(f"\nPure Time RANDOM Sampling (on {TIME_INTERVAL}):")
print(f"  Finite condition numbers: {len(d_time_rand_finite)}")
if len(kappa_time_rand_finite) > 0:
    print(f"  Min mean kappa: {np.min(kappa_time_rand_finite):.4e} at d = {d_time_rand_finite[np.argmin(kappa_time_rand_finite)]}")
    print(f"  Max mean kappa: {np.max(kappa_time_rand_finite):.4e} at d = {d_time_rand_finite[np.argmax(kappa_time_rand_finite)]}")

print(f"\nMixed RANDOM Sampling (time on {TIME_INTERVAL}, freq on {FREQ_INTERVAL}):")
print(f"  Finite condition numbers: {len(d_mixed_rand_finite)}")
if len(kappa_mixed_rand_finite) > 0:
    print(f"  Min mean kappa: {np.min(kappa_mixed_rand_finite):.4e} at d = {d_mixed_rand_finite[np.argmin(kappa_mixed_rand_finite)]}")
    print(f"  Max mean kappa: {np.max(kappa_mixed_rand_finite):.4e} at d = {d_mixed_rand_finite[np.argmax(kappa_mixed_rand_finite)]}")

# Comparison table
print(f"\nComparison at key dimensions (Random vs Regular):")
print(f"{'d':>6} | {'Pure REG':>12} | {'Pure RAND mean':>14} | {'Pure RAND std':>14} | {'Mixed REG':>12} | {'Mixed RAND mean':>15} | {'Mixed RAND std':>14}")
print("-"*110)

key_dims = [d for d in [1, 2, 5, 10, 20, 30] if d in dimensions]
for kd in key_dims:
    idx = np.where(d_vals == kd)[0]
    idx_rand = np.where(d_vals_rand == kd)[0]
    if len(idx) > 0 and len(idx_rand) > 0:
        kt_reg = kappa_time_vals[idx[0]]
        km_reg = kappa_mixed_vals[idx[0]]
        kt_rand = kappa_time_rand_mean[idx_rand[0]]
        km_rand = kappa_mixed_rand_mean[idx_rand[0]]
        
        kt_reg_str = f"{kt_reg:.2e}" if np.isfinite(kt_reg) else "inf"
        km_reg_str = f"{km_reg:.2e}" if np.isfinite(km_reg) else "inf"
        kt_rand_str = f"{kt_rand:.2e}" if np.isfinite(kt_rand) else "inf"
        km_rand_str = f"{km_rand:.2e}" if np.isfinite(km_rand) else "inf"
        
        kt_rand_std_val = kappa_time_rand_std[idx_rand[0]]
        km_rand_std_val = kappa_mixed_rand_std[idx_rand[0]]
        kt_std_str = f"{kt_rand_std_val:.2e}" if np.isfinite(kt_rand_std_val) else "nan"
        km_std_str = f"{km_rand_std_val:.2e}" if np.isfinite(km_rand_std_val) else "nan"
        print(f"{kd:6d} | {kt_reg_str:>12} | {kt_rand_str:>14} | {kt_std_str:>14} | {km_reg_str:>12} | {km_rand_str:>15} | {km_std_str:>14}")

### Random Sampling: Ratio Analysis

In [ ]:
# Compute ratio for random sampling where both are finite
both_rand_finite = time_rand_finite_mask & mixed_rand_finite_mask
d_both_rand = d_vals_rand[both_rand_finite]
ratio_rand = kappa_time_rand_mean[both_rand_finite] / kappa_mixed_rand_mean[both_rand_finite]

# --- Plot 1: Standalone random ratio ---
fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.semilogy(d_both_rand, ratio_rand, 'm.-', markersize=3, linewidth=0.8, 
             label=f'Random sampling (mean, n={num_trials_used})')
ax1.axhline(y=1, color='k', linestyle='--', alpha=0.5, label='Ratio = 1 (equal)')
ax1.set_xlabel('Dimension d', fontsize=12)
ax1.set_ylabel(r'$\kappa_2(A_{\mathrm{time}}^{\mathrm{rand}}) / \kappa_2(A_{\mathrm{mixed}}^{\mathrm{rand}})$', fontsize=12)
ax1.set_title(f'Sinc Basis (Random): Condition Number Ratio — Pure Time / Mixed\n(TIME_RATIO={TIME_RATIO}, time={TIME_INTERVAL}, freq={FREQ_INTERVAL})', fontsize=12)
ax1.grid(True, which='both', alpha=0.3)
ax1.legend()
plt.tight_layout()
plt.savefig('results/sinc_sampling_condition/sinc_condition_ratio_random.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nRatio statistics for RANDOM sampling (pure time / mixed), averaged over {num_trials_used} trials:")
print(f"  Min ratio: {np.min(ratio_rand):.4e} at d = {d_both_rand[np.argmin(ratio_rand)]}")
print(f"  Max ratio: {np.max(ratio_rand):.4e} at d = {d_both_rand[np.argmax(ratio_rand)]}")
print(f"  Mean ratio: {np.mean(ratio_rand):.4e}")
print(f"  Median ratio: {np.median(ratio_rand):.4e}")

### Ratio Comparison: Regular vs Random

In [ ]:
# --- Plot: Combined ratio comparison (Regular vs Random) ---
fig, ax = plt.subplots(figsize=(10, 5))
ax.semilogy(d_both, ratio, 'g-', markersize=2, linewidth=1.2, label='Regular sampling ratio')
ax.semilogy(d_both_rand, ratio_rand, 'm--', markersize=2, linewidth=1.2, 
            label=f'Random sampling ratio (mean, n={num_trials_used})')
ax.axhline(y=1, color='k', linestyle='--', alpha=0.5, label='Ratio = 1')
ax.set_xlabel('Dimension d', fontsize=12)
ax.set_ylabel(r'$\kappa_2(A_{\mathrm{time}}) / \kappa_2(A_{\mathrm{mixed}})$', fontsize=12)
ax.set_title(f'Sinc Basis: Ratio Comparison — Regular vs Random\n(TIME_RATIO={TIME_RATIO}, time={TIME_INTERVAL}, freq={FREQ_INTERVAL})', fontsize=12)
ax.grid(True, which='both', alpha=0.3)
ax.legend()

plt.tight_layout()
plt.savefig('results/sinc_sampling_condition/sinc_condition_ratio_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nFor comparison, REGULAR sampling ratio statistics:")
print(f"  Min ratio: {np.min(ratio):.4e} at d = {d_both[np.argmin(ratio)]}")
print(f"  Max ratio: {np.max(ratio):.4e} at d = {d_both[np.argmax(ratio)]}")
print(f"  Mean ratio: {np.mean(ratio):.4e}")
print(f"  Median ratio: {np.median(ratio):.4e}")

# =============================================================================
# EXPERIMENT 2b: Random Sampling with d-Dependent Shifts and Time Interval
# =============================================================================
# Same d-dependent shifts and time intervals as Experiment 1b,
# but with random (uniform) sample locations instead of equispaced grids.
# Results are averaged over `NUM_RANDOM_TRIALS` trials.

In [ ]:
results_ddep_random = []
output_file_ddep_random = Path("results/sinc_sampling_condition/sinc_condition_numbers_ddep_random.json")

print("Computing condition numbers: d-DEPENDENT shifts + RANDOM sampling...")
print(f"  Averaging over {NUM_RANDOM_TRIALS} trials (seed={RANDOM_SEED})")
print(f"  Shifts: n_0(d) = ceil(-d/2+1), time interval = [ceil(-d/2+1), floor(d/2)]")
print(f"  FREQ_INTERVAL = {FREQ_INTERVAL} (fixed)")
print(f"  TIME_RATIO = {TIME_RATIO}")
print("="*90)

for i, d in enumerate(dimensions):
    shift_start_d = int(np.ceil(-d / 2 + 1))
    shift_end_d = shift_start_d + d - 1
    time_interval_d = (float(shift_start_d), float(shift_end_d))
    k, m = compute_time_freq_split(d, TIME_RATIO)

    kappa_time_trials = []
    kappa_mixed_trials = []

    for trial in range(NUM_RANDOM_TRIALS):
        trial_offset = trial * 100000

        rng_pure = np.random.default_rng(RANDOM_SEED + d + trial_offset + 50000)
        rng_mixed_time = np.random.default_rng(RANDOM_SEED + d + trial_offset + 60000)
        rng_mixed_freq = np.random.default_rng(RANDOM_SEED + d + trial_offset + 70000)

        A_time_rand = build_pure_time_matrix_random(d, time_interval_d, shift_start_d, rng_pure)
        kappa_time_rand, _, _ = compute_condition_number(A_time_rand, d)

        A_mixed_rand, _, _ = build_mixed_matrix_random(d, TIME_RATIO, time_interval_d, FREQ_INTERVAL,
                                                        shift_start_d, rng_mixed_time, rng_mixed_freq)
        kappa_mixed_rand, _, _ = compute_condition_number(A_mixed_rand, d)

        kappa_time_trials.append(kappa_time_rand if np.isfinite(kappa_time_rand) else np.inf)
        kappa_mixed_trials.append(kappa_mixed_rand if np.isfinite(kappa_mixed_rand) else np.inf)

    kappa_time_trials = np.array(kappa_time_trials)
    kappa_mixed_trials = np.array(kappa_mixed_trials)

    ft = kappa_time_trials[np.isfinite(kappa_time_trials)]
    fm = kappa_mixed_trials[np.isfinite(kappa_mixed_trials)]

    result = {
        "d": d,
        "shift_start": shift_start_d,
        "shift_end": shift_end_d,
        "time_interval": list(time_interval_d),
        "freq_interval": list(FREQ_INTERVAL),
        "time_ratio": TIME_RATIO,
        "k": k, "m": m,
        "num_trials": NUM_RANDOM_TRIALS,
        "kappa_time_mean": float(np.mean(ft)) if len(ft) > 0 else "inf",
        "kappa_time_std": float(np.std(ft)) if len(ft) > 1 else 0.0,
        "kappa_mixed_mean": float(np.mean(fm)) if len(fm) > 0 else "inf",
        "kappa_mixed_std": float(np.std(fm)) if len(fm) > 1 else 0.0,
        "num_finite_time_trials": len(ft),
        "num_finite_mixed_trials": len(fm),
        "random_seed": RANDOM_SEED
    }
    results_ddep_random.append(result)

    if (i + 1) % 20 == 0 or d in [1, 10, 100, 500]:
        kt_str = f"{result['kappa_time_mean']:.4e}" if result['kappa_time_mean'] != "inf" else "inf"
        km_str = f"{result['kappa_mixed_mean']:.4e}" if result['kappa_mixed_mean'] != "inf" else "inf"
        print(f"[{i+1:3d}/{len(dimensions)}] d={d:5d}, shifts=[{shift_start_d}..{shift_end_d}]: "
              f"mean kappa_time={kt_str}, mean kappa_mixed={km_str}")

with open(output_file_ddep_random, 'w') as f:
    json.dump(results_ddep_random, f, indent=2)

print("="*90)
print(f"Saved results to {output_file_ddep_random}")

In [ ]:
# Extract d-dependent random data for plotting
d_vals_ddep_rand = np.array([r['d'] for r in results_ddep_random])
kappa_time_ddep_rand_mean = np.array([
    r['kappa_time_mean'] if r['kappa_time_mean'] != 'inf' else np.inf for r in results_ddep_random])
kappa_mixed_ddep_rand_mean = np.array([
    r['kappa_mixed_mean'] if r['kappa_mixed_mean'] != 'inf' else np.inf for r in results_ddep_random])
kappa_time_ddep_rand_std = np.array([
    r.get('kappa_time_std', 0.0) if r.get('kappa_time_std', 'nan') != 'nan' else 0.0 for r in results_ddep_random])
kappa_mixed_ddep_rand_std = np.array([
    r.get('kappa_mixed_std', 0.0) if r.get('kappa_mixed_std', 'nan') != 'nan' else 0.0 for r in results_ddep_random])

time_ddep_rand_finite = np.isfinite(kappa_time_ddep_rand_mean)
mixed_ddep_rand_finite = np.isfinite(kappa_mixed_ddep_rand_mean)

d_t_ddr_fin = d_vals_ddep_rand[time_ddep_rand_finite]
k_t_ddr_fin = kappa_time_ddep_rand_mean[time_ddep_rand_finite]
k_t_ddr_std_fin = kappa_time_ddep_rand_std[time_ddep_rand_finite]
d_m_ddr_fin = d_vals_ddep_rand[mixed_ddep_rand_finite]
k_m_ddr_fin = kappa_mixed_ddep_rand_mean[mixed_ddep_rand_finite]
k_m_ddr_std_fin = kappa_mixed_ddep_rand_std[mixed_ddep_rand_finite]

print(f"d-dep random: Pure time finite: {len(d_t_ddr_fin)}, Mixed finite: {len(d_m_ddr_fin)} out of {len(d_vals_ddep_rand)}")

# --- Plot: Randomized pure time vs randomized mixed (d-dependent) ---
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

ax.semilogy(d_t_ddr_fin, k_t_ddr_fin, 'b.-', markersize=3, linewidth=0.8,
            label=f'Pure time RANDOM (mean, n={NUM_RANDOM_TRIALS})')
if np.any(k_t_ddr_std_fin > 0):
    ax.fill_between(d_t_ddr_fin,
                    np.maximum(k_t_ddr_fin - k_t_ddr_std_fin, 1e-10),
                    k_t_ddr_fin + k_t_ddr_std_fin, alpha=0.2, color='blue')

ax.semilogy(d_m_ddr_fin, k_m_ddr_fin, 'r.-', markersize=3, linewidth=0.8,
            label=f'Mixed RANDOM (mean, n={NUM_RANDOM_TRIALS})')
if np.any(k_m_ddr_std_fin > 0):
    ax.fill_between(d_m_ddr_fin,
                    np.maximum(k_m_ddr_fin - k_m_ddr_std_fin, 1e-10),
                    k_m_ddr_fin + k_m_ddr_std_fin, alpha=0.2, color='red')

ax.set_xlabel('Dimension d', fontsize=12)
ax.set_ylabel(r'$\kappa_2(A^{(d)})$', fontsize=12)
ax.set_title(r'EXPERIMENT 2b: Random Sampling with d-Dependent Shifts $[\lceil -d/2+1 \rceil \ldots \lfloor d/2 \rfloor]$'
             f'\nfreq interval = {FREQ_INTERVAL}, avg over {NUM_RANDOM_TRIALS} trials', fontsize=12)
ax.legend(loc='upper left', fontsize=10)
ax.grid(True, which='both', alpha=0.3)
ax.set_xlim([0, max(dimensions) * 1.02])

plt.tight_layout()
plt.savefig('results/sinc_sampling_condition/sinc_condition_number_ddep_random.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Saved plot to results/sinc_sampling_condition/sinc_condition_number_ddep_random.png")

# Summary table
print(f"\nExperiment 2b: d-dependent shifts + random sampling summary:")
print(f"{'d':>6} | {'n_0':>5} | {'n_end':>5} | {'time_int':>18} | {'mean kappa_time':>16} | {'mean kappa_mixed':>16}")
print("-"*90)
for kd in [1, 2, 5, 10, 15, 20, 25, 30]:
    idx = np.where(d_vals_ddep_rand == kd)[0]
    if len(idx) > 0:
        r = results_ddep_random[idx[0]]
        kt = kappa_time_ddep_rand_mean[idx[0]]
        km = kappa_mixed_ddep_rand_mean[idx[0]]
        kt_str = f"{kt:.4e}" if np.isfinite(kt) else "inf"
        km_str = f"{km:.4e}" if np.isfinite(km) else "inf"
        print(f"{kd:6d} | {r['shift_start']:5d} | {r['shift_end']:5d} | {str(r['time_interval']):>18} | {kt_str:>16} | {km_str:>16}")

In [ ]:
# Extract d-dependent random data for plotting
d_vals_ddep_rand = np.array([r['d'] for r in results_ddep_random])

kappa_time_ddep_rand_mean = np.array([
    r['kappa_time_mean'] if r['kappa_time_mean'] != 'inf' else np.inf
    for r in results_ddep_random
])

kappa_mixed_ddep_rand_mean = np.array([
    r['kappa_mixed_mean'] if r['kappa_mixed_mean'] != 'inf' else np.inf
    for r in results_ddep_random
])

time_ddep_rand_finite = np.isfinite(kappa_time_ddep_rand_mean)
mixed_ddep_rand_finite = np.isfinite(kappa_mixed_ddep_rand_mean)

d_t_ddr_fin = d_vals_ddep_rand[time_ddep_rand_finite]
k_t_ddr_fin = kappa_time_ddep_rand_mean[time_ddep_rand_finite]

d_m_ddr_fin = d_vals_ddep_rand[mixed_ddep_rand_finite]
k_m_ddr_fin = kappa_mixed_ddep_rand_mean[mixed_ddep_rand_finite]

print(f"d-dep random: Pure time finite: {len(d_t_ddr_fin)}, Mixed finite: {len(d_m_ddr_fin)} out of {len(d_vals_ddep_rand)}")

# --- Plot: Randomized pure time vs randomized mixed (d-dependent) ---
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

ax.semilogy(
    d_t_ddr_fin, k_t_ddr_fin, 'b.-',
    markersize=10, linewidth=1.5,
    label=f'Pure time sampling with uniform randomization'
)

ax.semilogy(
    d_m_ddr_fin, k_m_ddr_fin, 'r.-',
    markersize=10, linewidth=1.5,
    label=f'Two-sided sampling with uniform randomization'
)

ax.set_xlabel('Dimension $N$', fontsize=23)
ax.set_ylabel(r'Condition number', fontsize=23)
# ax.set_title(
#     r'd-dimensional bandlimited spaces: random Sampling with d-Dependent Shifts $[\lceil -d/2+1 \rceil \ldots \lfloor d/2 \rfloor]$'
#     f'\nFrequency interval = {FREQ_INTERVAL}',
#     fontsize=12
# )
ax.legend(loc='upper left', fontsize=14)
ax.grid(True, which='both', alpha=0.3)
ax.set_xlim([0, max(dimensions) * 1.02])

ax.tick_params(axis='x', labelsize=18)  # x-axis tick label size
ax.tick_params(axis='y', labelsize=12)  # y-axis tick label size

plt.tight_layout()
plt.savefig('results/sinc_sampling_condition/sinc_condition_number_ddep_random.png', dpi=150, bbox_inches='tight')
plt.show()

print("Saved plot to results/sinc_sampling_condition/sinc_condition_number_ddep_random.png")

# Summary table
print(f"d-dimensional bandlimited spaces: d-dependent shifts + random sampling:")
print(f"{'d':>6} | {'n_0':>5} | {'n_end':>5} | {'time_int':>18} | {'mean kappa_time':>16} | {'mean kappa_mixed':>16}")
print("-" * 90)

for kd in [1, 2, 5, 10, 15, 20, 25, 30]:
    idx = np.where(d_vals_ddep_rand == kd)[0]
    if len(idx) > 0:
        r = results_ddep_random[idx[0]]
        kt = kappa_time_ddep_rand_mean[idx[0]]
        km = kappa_mixed_ddep_rand_mean[idx[0]]
        kt_str = f"{kt:.4e}" if np.isfinite(kt) else "inf"
        km_str = f"{km:.4e}" if np.isfinite(km) else "inf"
        print(f"{kd:6d} | {r['shift_start']:5d} | {r['shift_end']:5d} | {str(r['time_interval']):>18} | {kt_str:>16} | {km_str:>16}")